In [1]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o-mini")

from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="너는 미녀와 야수에 나오는 미녀야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕? 저는 개스톤입니다. 오늘 시간 괜찮으시면 저녁 같이 먹을까요?"),
]

model.invoke(messages)

AIMessage(content='안녕하세요, 개스톤! 당신의 제안은 정말 매력적인 것 같아요. 하지만 저는 아름다움보다 진정한 사랑과 이해를 중요시하거든요. 저녁 함께하는 것은 생각해볼 수 있지만, 제 마음을 이해해 주실 수 있을까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 62, 'total_tokens': 124, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-C3YS8p9LTigquzhLoZ1BeBysvENkH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--cd0c9262-b268-4ef8-a242-3ba95113c5b4-0', usage_metadata={'input_tokens': 62, 'output_tokens': 62, 'total_tokens': 124, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

result = model.invoke(messages)
parser.invoke(result)

'안녕하세요, 개스톤! 정말 친절하게도 저를 초대해 주셨네요. 하지만, 저는 친구들과 함께한 소중한 순간들과 야수와의 특별한 관계를 소중히 여기고 있어요. 그렇지만 이렇게 초대해 주신 것은 정말 감사해요! 다른 날에는 좋겠죠?'

In [3]:
chain = model | parser
chain.invoke(messages)

'안녕하세요, 개스톤! 당신의 제안은 정말 고맙지만, 저는 아직 그런 생각을 하지는 않았어요. 다른 사람들과의 멋진 시간을 소중히 여기지만, 제 마음은 조금 다르답니다. 취향이나 기호를 이해해 주시면 좋겠어요. 당신은 저에게 특별한 친구 같은 존재지만, 이렇게 주위를 배려해 주셔서 감사합니다. 다른 계획이 있다면 함께 이야기해볼까요?'

In [4]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {story}에 나오는 {character_a} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_b}입니다. 오늘 시간 괜찮으시면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

result = prompt_template.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

print(result)

messages=[SystemMessage(content='너는 미녀와 야수에 나오는 미녀 역할이다. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕? 저는 야수입니다. 오늘 시간 괜찮으시면 저녁 같이 할까요?', additional_kwargs={}, response_metadata={})]


In [5]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

'안녕하세요, 야수님! 저녁 함께하는 건 정말 좋은 생각이에요. 당신과의 저녁 식사는 언제나 특별하니까요. 어떤 음식을 드시고 싶으신가요?'

In [6]:
from typing import Literal
from pydantic import BaseModel, Field

class Adlib(BaseModel):
    """스토리 설정과 사용자 입력에 반응하는 대사를 만드는 클래스"""
    answer: str = Field(description="스토리 설정과 사용자와의 대화 기록에 따라 생성된 대사")
    main_emotion: Literal["기쁨", "분노", "슬픔", "공포", "냉소", "불쾌", "중립"] = Field(description="대사의 주요 감정")
    main_emotion_intensity: float = Field(description="대사의 주요 감정의 강도 (0.0 ~ 1.0)")

structured_llm = model.with_structured_output(Adlib)
adlib_chain = prompt_template | structured_llm

adlib_chain.invoke({
    "story": "미녀와 야수",
    "character_a": "벨",
    "character_b": "개스톤",
    "activity": "저녁"
})

Adlib(answer='안녕하세요, 개스톤! 저녁 제안해 주셔서 고마워요. 하지만 제가 현재 타인을 먼저 생각할 때 더 많은 제약이 있다는 걸 알고 계신가요?', main_emotion='중립', main_emotion_intensity=0.5)